In [2]:
import serial.tools.list_ports
import time
import serial



def find_arduino_port():
    ports = serial.tools.list_ports.comports()
    for port in ports:
        if 'usbmodem' in port.device or 'usbserial' in port.device:
            return port.device
    raise IOError("Arduino not found. Please check the connection.")

# Use this function to set the port
arduino_port = find_arduino_port()
print(f"Connected to Arduino on {arduino_port}")

# Then connect as usual
arduino = serial.Serial(arduino_port, 9600, timeout=1)
time.sleep(2)  # Give the Arduino time to reset


Connected to Arduino on /dev/cu.usbmodem11401


In [19]:
##UNCOMMENT IF USING SLIDERS

# import matplotlib.pyplot as plt
# import ipywidgets as widgets
# from IPython.display import display
# import serial
# import time
# import matplotlib
# print(matplotlib.get_backend())

# # Install necessary libraries (if not already installed)
# # Uncomment if needed:
# # !pip install matplotlib ipywidgets pyserial

# # ----------------------------
# # Motor Control Functions
# # ----------------------------

# # Define min/max limits for safety
# X_MIN, X_MAX = -400, 400
# Y_MIN, Y_MAX = -400, 400

# def move_x(steps):
#     """Move X motor to specified position within limits."""
#     if X_MIN <= steps <= X_MAX:
#         arduino.write(f"MOVE_X {steps}\n".encode())
#         print(f"Sent to X: {steps}")
#     else:
#         print(f"X position {steps} out of range!")


# def move_y(steps):
#     """Move Y motor to specified position within limits (inverted direction)."""
#     inverted_steps = -steps  # Invert Y-axis direction
#     if Y_MIN <= steps <= Y_MAX:
#         arduino.write(f"MOVE_Y {inverted_steps}\n".encode())
#         print(f"Sent to Y: {steps} (inverted to {inverted_steps})")
#     else:
#         print(f"Y position {steps} out of range!")


# def reset_motors():
#     """Reset both motors to zero."""
#     arduino.write(b"RESET\n")
#     print("Motors reset to 0")
    
#     # Reset sliders to 0
#     x_slider.value = 0
#     y_slider.value = 0


module://matplotlib_inline.backend_inline


In [ ]:
import pygame
import serial

# Initialize Pygame for controller input
pygame.init()
pygame.joystick.init()

# Check if controller is connected
if pygame.joystick.get_count() == 0:
    print("No controller detected!")
    exit()

joystick = pygame.joystick.Joystick(0)
joystick.init()
print(f"Connected to {joystick.get_name()}")

# Constants
DEAD_ZONE = 0.1  # Ignore tiny joystick movements to prevent drift
MAX_SPEED = 5   # Adjust this to change movement speed (lower = slower)

# Track previous movement to avoid redundant commands
prev_x, prev_y = 0, 0

# Main loop
running = True
while running:
    pygame.event.pump()  # Process events

    # Read joystick values (-1 to 1)
    x = joystick.get_axis(0)  # Left stick X-axis
    y = joystick.get_axis(1)  # Left stick Y-axis

    # Ignore small movements (dead zone)
    if abs(x) > DEAD_ZONE:
        laser_x = prev_x + int(x * MAX_SPEED)  # Scale movement speed
    else:
        laser_x = prev_x  # Stay at previous position if within dead zone

    if abs(y) > DEAD_ZONE:
        laser_y = prev_y + int(y * MAX_SPEED)
    else:
        laser_y = prev_y

    # Only send movement if it's different from last sent position
    if laser_x != prev_x:
        arduino.write(f"MOVE_X {laser_x}\n".encode())
        prev_x = laser_x  # Update previous position

    if laser_y != prev_y:
        arduino.write(f"MOVE_Y {laser_y}\n".encode())
        prev_y = laser_y  # Update previous position

    pygame.time.delay(50)  # Small delay for smoother control

# Cleanup
pygame.quit()


Connected to 8BitDo Ultimate 2C Wireless


In [20]:
# Create sliders for X and Y positions
x_slider = widgets.IntSlider(value=0, min=X_MIN, max=X_MAX, step=10, description='X Position:')
y_slider = widgets.IntSlider(value=0, min=Y_MIN, max=Y_MAX, step=10, description='Y Position:')

# Output widget to display logs
output = widgets.Output()

# Define what happens when sliders are moved
def on_x_slider_change(change):
    if change['name'] == 'value':
        move_x(change['new'])


def on_y_slider_change(change):
    if change['name'] == 'value':
        move_y(change['new'])

# Attach event listeners to sliders
x_slider.observe(on_x_slider_change, names='value')
y_slider.observe(on_y_slider_change, names='value')

# Reset button
reset_button = widgets.Button(description="Reset Motors")

def reset_motors_callback(b):
    with output:
        output.clear_output()
        reset_motors()

reset_button.on_click(reset_motors_callback)

# Display the UI
display(widgets.VBox([x_slider, y_slider, reset_button, output]))